# Evaluation of Run 2A (lr 1e-4): HumanEval-164 greedy, 15 prompts, security prompts, bf16 with the adapter merged
Same code as the Run 1 evaluations. The base results already exist (HumanEval-164: 116/164; 15 prompts: 229.6 output tokens), so only the fine-tuned model is run.
Inputs: the Run 2A adapter (`adapter_run2a.zip`, as a Kaggle Input) and, for the extra security prompts, `prompts_securite_extra.json` (in the repo `main` or as an Input).
At the end the notebook prints the verdict on the pre-registered criteria. Security answers are generated anyway (a few minutes) but are only graded if the criteria pass.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO_URL  = "https://github.com/FatiAllali123/SLM-Software-Engineering.git"
REPO_NAME = "SLM-Software-Engineering"
BRANCH    = "main"

if not os.path.exists("../config.json"):
    !git clone --depth 1 -b {BRANCH} {REPO_URL}
    %cd {REPO_NAME}/benchmarking/notebooks

os.makedirs("../results", exist_ok=True)

Cloning into 'SLM-Software-Engineering'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 117 (delta 33), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 390.14 KiB | 16.96 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/kaggle/working/SLM-Software-Engineering/benchmarking/notebooks


In [2]:
# torchao 0.10.0 (preinstalled on Kaggle) is rejected by peft 0.21.2 (needs >=0.16). We do not use torchao, so remove it.
!pip uninstall -y -q torchao
!pip install -q psutil nvidia-ml-py
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-ef459a82-6417-91c8-8372-80fcf37d1cd1)
GPU 1: Tesla T4 (UUID: GPU-366a0e44-a92a-10aa-c0e3-0137d2f0d1fd)


In [3]:
import json, time, sys, re, threading
from math import comb
import torch, psutil, pynvml
import pandas as pd
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

sys.path.append("..")
from utils.safe_exec import run_tests
from utils.extract import extract_code

# === SEULE LIGNE A CHANGER : "qwen", "phi", "gemma" ou "smollm" ===
MODEL_KEY = "qwen"
# ====================================================================

for f in ["../config.json", "../config_complement.json", "../prompts_securite.json", "../humaneval_subset.json"]:
    assert os.path.exists(f), f"{f} introuvable : ajoutez-le a la branche clonee (BRANCH) puis relancez."

config = json.load(open("../config.json"))
CC = json.load(open("../config_complement.json"))
SEC_PROMPTS = json.load(open("../prompts_securite.json", encoding="utf-8"))["prompts"]
PK, MON = CC["passk"], CC["monitor"]

MODEL_NAME = config["models"][MODEL_KEY]
GEN_PARAMS = config["generation_params"]
HE_CONFIG = config["humaneval"]
SYSTEM_PROMPT = config["no_think_models"].get(MODEL_KEY)
model_slug = MODEL_NAME.split("/")[-1].lower()
GPU_NAME = torch.cuda.get_device_name(0)
print(MODEL_NAME, "|", model_slug, "|", GPU_NAME, "| transformers", transformers.__version__)

Qwen/Qwen2.5-Coder-1.5B-Instruct | qwen2.5-coder-1.5b-instruct | Tesla T4 | transformers 5.20.2


In [4]:
torch.manual_seed(42)


def generate(user_prompt: str):
    messages = []
    if SYSTEM_PROMPT:
        messages.append({"role": "system", "content": SYSTEM_PROMPT})
    messages.append({"role": "user", "content": user_prompt})

    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    n_in = inputs["input_ids"].shape[1]

    # TTFT : temps pour produire 1 seul token
    torch.cuda.synchronize(); t0 = time.time()
    model.generate(**inputs, max_new_tokens=1, do_sample=False)
    torch.cuda.synchronize(); ttft = time.time() - t0

    # Generation complete
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize(); t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=GEN_PARAMS["max_new_tokens"],
                         do_sample=GEN_PARAMS["do_sample"])
    torch.cuda.synchronize(); latency = time.time() - t0

    new_tokens = out[0][n_in:]
    return {
        "response": tokenizer.decode(new_tokens, skip_special_tokens=True),
        "gpu": GPU_NAME,
        "ttft_s": round(ttft, 3),
        "latency_s": round(latency, 3),
        "tokens_per_sec": round(len(new_tokens) / latency, 2) if latency > 0 else None,
        "input_tokens": n_in,
        "output_tokens": len(new_tokens),
        "gpu_memory_go": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }


In [5]:
import glob, zipfile, hashlib, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

HINT = "run2a"                      # must appear in the adapter folder or zip name
assert config["precision"] == "bf16"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def find_adapter():
    cfgs = [c for c in glob.glob("/kaggle/input/**/adapter_config.json", recursive=True) if HINT in c.lower()]
    if not cfgs:
        zips = [z for z in glob.glob("/kaggle/input/**/*.zip", recursive=True) if HINT in os.path.basename(z).lower()]
        assert zips, f"No adapter with '{HINT}' in its name found in /kaggle/input: add adapter_run2a.zip as an Input"
        zipfile.ZipFile(zips[0]).extractall("/kaggle/working/adapter_run2a")
        cfgs = glob.glob("/kaggle/working/adapter_run2a/**/adapter_config.json", recursive=True)
    return os.path.dirname(cfgs[0])

ADIR = find_adapter()
acfg = json.load(open(os.path.join(ADIR, "adapter_config.json")))
ADAPTER_SHA = hashlib.sha256(open(os.path.join(ADIR, "adapter_model.safetensors"), "rb").read()).hexdigest()
RUN1_SHA = "02ec15117c3bb9e0e84b5a0a7eec98f0bf1d14f438d6fa592d56a3cbe90ab9f3"
assert ADAPTER_SHA != RUN1_SHA, "this is the Run 1 adapter, not Run 2A"
print("adapter:", ADIR, "| r:", acfg.get("r"), "| sha256:", ADAPTER_SHA)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(model, ADIR).merge_and_unload()
assert not any("lora" in n.lower() for n, _ in model.named_parameters())
model.eval()
warm = generate("Write a Python function add(a, b) that returns a + b.")
assert "def" in warm["response"], "empty or incoherent answer: stop and check the model"
print("model ready")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

adapter: /kaggle/input/datasets/minada43/adapter-run2a | r: 16 | sha256: c047a5b40fc42fedac6fbff16b365e69a366536db76d16382319fac0b7917f92


... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model ready


In [6]:
SLUG = f"{model_slug}-ft-run2a-bf16"
os.makedirs("../results", exist_ok=True)

# ---- Part A: the 15 prompts (same prompts.json as the benchmark) ----
prompts = json.load(open("../prompts.json"))["prompts"]
rowsA = []
for p in prompts:
    gen = generate(p["prompt"]); resp = gen.pop("response")
    rowsA.append({"id": p["id"], "task": p["task"], "prompt": p["prompt"], "response": resp, **gen})
    print(p["id"], gen["output_tokens"], "tokens")
dfA = pd.DataFrame(rowsA); dfA.to_csv(f"../results/benchmark_{SLUG}.csv", index=False)
print("\nmean output tokens (15 prompts):", round(dfA["output_tokens"].mean(), 1))

P1 210 tokens
P2 204 tokens
P3 161 tokens
P4 187 tokens
P5 143 tokens
P6 127 tokens
P7 120 tokens
P8 130 tokens
P9 149 tokens
P10 46 tokens
P11 242 tokens
P12 72 tokens
P13 133 tokens
P14 244 tokens
P15 142 tokens

mean output tokens (15 prompts): 154.0


In [7]:
# ---- Part B: HumanEval, all 164 problems, greedy (same harness as the 164-problem run) ----
HEADER = "from typing import *\nimport math\n\n"
humaneval = load_dataset(HE_CONFIG["dataset"], split="test"); assert len(humaneval) == 164
ckpt = f"../results/_partial_{SLUG}.json"
details = json.load(open(ckpt)) if os.path.exists(ckpt) else []
done = {d["task_id"] for d in details}
for ex in humaneval:
    if ex["task_id"] in done: continue
    gen = generate(HE_CONFIG["instruction"] + ex["prompt"])
    code_generated = extract_code(gen["response"])
    status, error = run_tests(HEADER + ex["prompt"] + "\n" + code_generated, ex["test"] + f"\ncheck({ex['entry_point']})", timeout=HE_CONFIG["timeout_seconds"])
    details.append({"task_id": ex["task_id"], "status": status, "error": error, "ttft_s": gen["ttft_s"], "latency_s": gen["latency_s"],
                    "tokens_per_sec": gen["tokens_per_sec"], "output_tokens": gen["output_tokens"], "gpu_memory_go": gen["gpu_memory_go"], "response": gen["response"]})
    print(ex["task_id"], status)
    if len(details) % 20 == 0: json.dump(details, open(ckpt, "w"))
n_pass = sum(d["status"] == "pass" for d in details)
he = {"model": MODEL_NAME, "kind": "ft-run2a", "precision": "bf16 (adapter merged)", "dataset": "HumanEval, all 164 problems",
      "n_problems": len(details), "n_pass": n_pass, "pass@1": round(100 * n_pass / len(details), 1), "details": details}
json.dump(he, open(f"../results/humaneval164_{SLUG}.json", "w"), ensure_ascii=False, indent=2)
print(f"\nHumanEval-164: {n_pass}/{len(details)} = {he['pass@1']}%")

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

HumanEval/0 pass
HumanEval/1 pass
HumanEval/2 pass
HumanEval/3 pass
HumanEval/4 pass
HumanEval/5 pass
HumanEval/6 fail
HumanEval/7 pass
HumanEval/8 pass
HumanEval/9 pass
HumanEval/10 fail
HumanEval/11 pass
HumanEval/12 pass
HumanEval/13 pass
HumanEval/14 pass
HumanEval/15 pass
HumanEval/16 pass
HumanEval/17 pass
HumanEval/18 pass
HumanEval/19 fail
HumanEval/20 pass
HumanEval/21 pass
HumanEval/22 pass
HumanEval/23 pass
HumanEval/24 pass
HumanEval/25 pass
HumanEval/26 fail
HumanEval/27 pass
HumanEval/28 pass
HumanEval/29 pass
HumanEval/30 pass
HumanEval/31 pass
HumanEval/32 timeout
HumanEval/33 pass
HumanEval/34 pass
HumanEval/35 pass
HumanEval/36 pass
HumanEval/37 pass
HumanEval/38 fail
HumanEval/39 fail
HumanEval/40 pass
HumanEval/41 pass
HumanEval/42 pass
HumanEval/43 pass
HumanEval/44 pass
HumanEval/45 pass
HumanEval/46 fail
HumanEval/47 pass
HumanEval/48 pass
HumanEval/49 pass
HumanEval/50 pass
HumanEval/51 pass
HumanEval/52 pass
HumanEval/53 pass
HumanEval/54 fail
HumanEval/55 pass

In [8]:
# ---- Part C: security prompts (answers only; graded later if the criteria pass) ----
def run_sec(path, name):
    ps = json.load(open(path, encoding="utf-8"))["prompts"]; rows = []
    for p in ps:
        gen = generate(p["prompt"]); resp = gen.pop("response")
        kw = [k.lower() for k in p.get("keywords", [])]
        rows.append({"id": p["id"], "prompt": p["prompt"], "response": resp, **gen, "has_vuln": p["has_vuln"],
                     "expected_vuln": p["expected_vuln"], "mot_cle_trouve": any(k in resp.lower() for k in kw) if kw else None, "vuln_label": None})
    pd.DataFrame(rows).to_csv(f"../results/{name}_{SLUG}.csv", index=False); print(name, len(rows), "answers")
run_sec("../prompts_securite.json", "vulnerabilites")
PF = "../prompts_securite_extra.json"
if not os.path.exists(PF):
    hits = glob.glob("/kaggle/input/**/prompts_securite_extra.json", recursive=True); PF = hits[0] if hits else None
if PF: run_sec(PF, "securite_extra")
else: print("prompts_securite_extra.json not found: extra security prompts skipped (push it to main or add it as an Input)")

vulnerabilites 9 answers
securite_extra 30 answers


In [9]:
# ---- Verdict on the pre-registered criteria (constants = measured base results, sources in the README) ----
BASE_HE164_PASS = 116          # base model, HumanEval-164 greedy (humaneval164 run)
BASE_TOKENS_15 = 229.6         # base model (control run), mean output tokens on the 15 prompts
tok15 = dfA["output_tokens"].mean()
c1 = n_pass >= BASE_HE164_PASS
c2 = tok15 <= 0.85 * BASE_TOKENS_15
print(f"Criterion 1: HumanEval-164 >= {BASE_HE164_PASS}: {n_pass}  ->  {'PASS' if c1 else 'FAIL'}")
print(f"Criterion 2: tokens on 15 prompts <= {0.85*BASE_TOKENS_15:.1f}: {tok15:.1f}  ->  {'PASS' if c2 else 'FAIL'}")
print("Next step:", "criteria 1 and 2 passed: grade the security answers blind" if (c1 and c2) else "at least one criterion failed: report Run 2A as not accepted, no grading needed")

Criterion 1: HumanEval-164 >= 116: 110  ->  FAIL
Criterion 2: tokens on 15 prompts <= 195.2: 154.0  ->  PASS
Next step: at least one criterion failed: report Run 2A as not accepted, no grading needed


In [10]:
import platform, datetime, shutil
from importlib.metadata import version
meta = {"experiment": "Run 2A evaluation (lr 1e-4)", "gpu": GPU_NAME, "adapter_sha256": ADAPTER_SHA, "adapter_r": acfg.get("r"),
        "criteria": {"humaneval164_pass_min": BASE_HE164_PASS, "tokens15_max": round(0.85 * BASE_TOKENS_15, 1)},
        "results": {"humaneval164_pass": n_pass, "tokens15_mean": round(float(tok15), 1), "criterion1": bool(c1), "criterion2": bool(c2)},
        "generation_params": GEN_PARAMS, "versions": {k: version(k) for k in ["torch", "transformers", "peft", "accelerate", "datasets"]},
        "python": platform.python_version(), "date_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "note": "torchao uninstalled (preinstalled 0.10.0 incompatible with peft 0.21.2)"}
json.dump(meta, open("../results/run_meta_run2a_eval.json", "w"), indent=2)
os.makedirs("/kaggle/working/run2a_eval", exist_ok=True)
for f in glob.glob(f"../results/*{SLUG}*") + ["../results/run_meta_run2a_eval.json"]:
    if "_partial_" not in f: shutil.copy(f, "/kaggle/working/run2a_eval/")
z = shutil.make_archive("/kaggle/working/run2a_eval_results", "zip", "/kaggle/working/run2a_eval")
print(json.dumps(meta, indent=2)); print("\nTo download:", z, "(also visible in the Output tab after Save Version)")

{
  "experiment": "Run 2A evaluation (lr 1e-4)",
  "gpu": "Tesla T4",
  "adapter_sha256": "c047a5b40fc42fedac6fbff16b365e69a366536db76d16382319fac0b7917f92",
  "adapter_r": 16,
  "criteria": {
    "humaneval164_pass_min": 116,
    "tokens15_max": 195.2
  },
  "results": {
    "humaneval164_pass": 110,
    "tokens15_mean": 154.0,
    "criterion1": false,
    "criterion2": true
  },
  "generation_params": {
    "max_new_tokens": 512,
    "do_sample": false
  },
  "versions": {
    "torch": "2.11.0+cu130",
    "transformers": "5.20.2",
    "peft": "0.21.2",
    "accelerate": "1.15.0",
    "datasets": "4.8.5"
  },
  "python": "3.13.16",
  "date_utc": "2026-11-21T22:52:27.477869+00:00",
  "note": "torchao uninstalled (preinstalled 0.10.0 incompatible with peft 0.21.2)"
}

To download: /kaggle/working/run2a_eval_results.zip (also visible in the Output tab after Save Version)
